# EGC COG Tree

Builds the COG annotation tree and writes one `.cog.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import requests
from pathlib import Path
from collections import defaultdict
from bs4 import BeautifulSoup
import sys
# The repo root: the nearest folder, from where this runs upward, that
# holds DataStructures/ -- so no path below is tied to one machine.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'DataStructures').is_dir())
sys.path.append(str(REPO / 'Notebooks'))
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc

## 1. Load sample list

In [2]:
with open(REPO / 'sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Fetch COG metadata from NIH

In [3]:
urls = [
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/J/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/A/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/K/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/L/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/B/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/D/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Y/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/V/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/T/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/M/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/N/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Z/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/W/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/U/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/O/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/X/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/C/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/G/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/E/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/F/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/H/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/I/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/P/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/Q/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/R/",
    "https://www.ncbi.nlm.nih.gov/research/cog/cogcategory/S/",
]

cogs = {}
cat_names = {}  # {letter: full category description, e.g. "Translation, ribosomal structure and biogenesis"}

# NCBI's own live page for category X currently renders a literal "nan"
# instead of a real description (confirmed against both the nav menu and
# the page's own <h1> -- an upstream data bug on their end, not a parsing
# issue on ours). X is well-established, stable COG classification, so
# this override is a known correction, not a guess. Only applied if the
# scrape actually produced "nan" -- if NCBI ever fixes their page, this
# stops firing on its own rather than silently overriding a corrected value.
KNOWN_CATEGORY_NAME_FIXES = {"X": "Mobilome: prophages, transposons"}

for url in urls:
    r = requests.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    cat_letter = url.rstrip("/").split("/")[-1]
    cogs[cat_letter] = {}

    # Each category page's own nav menu links to all 26 categories,
    # including itself -- the link matching this page's own URL carries
    # its full name, e.g. "J - Translation, ribosomal structure and
    # biogenesis". Reuses this same request instead of a separate fetch
    # or a hardcoded list.
    cat_link = soup.find("a", href=f"/research/cog/cogcategory/{cat_letter}/")
    if cat_link:
        cat_text = cat_link.get_text(strip=True)
        cat_name = cat_text.split(" - ", 1)[1] if " - " in cat_text else cat_text
        if cat_name.strip().lower() == "nan" and cat_letter in KNOWN_CATEGORY_NAME_FIXES:
            cat_name = KNOWN_CATEGORY_NAME_FIXES[cat_letter]
        cat_names[cat_letter] = cat_name

    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 5:
            continue
        link_tag = tds[2].find("a")
        if not link_tag:
            continue
        cog_id = link_tag.get_text(strip=True)
        annot  = tds[4].get_text(strip=True)
        cogs[cat_letter][cog_id] = [annot, link_tag.get("href")]

# NCBI lists many COGs on more than one category's page -- `cogs` above
# captures every (letter, cog_id) membership (it's keyed by letter on the
# outside), and cog_map keeps every category a COG belongs to when
# flattening it, not just whichever page happened to be scraped last.
cog_map: dict[str, dict] = {}
for cat, cog_dict in cogs.items():
    for cog_id, (name, _href) in cog_dict.items():
        entry = cog_map.setdefault(cog_id, {"name": name, "categories": []})
        entry["categories"].append(cat)

multi_category_count = sum(1 for e in cog_map.values() if len(e["categories"]) > 1)
print(f"cog_map: {len(cog_map)} entries ({multi_category_count} in more than one category)")
print(f"cat_names: {len(cat_names)} categories")
for letter, name in sorted(cat_names.items()):
    print(f"  {letter}: {name}")

# Save the full (letter, cog_id) membership table to disk -- mirrors
# ko00001.keg's role for KEGG: migrate_cog_paths_to_sql.py reads this
# directly to build cog_paths, instead of re-scraping NCBI (which this
# cell does live, over the network) or relying on annotation_catalog's
# single-valued parent column, which can only ever express one category
# per COG.
cog_categories_path = REPO / 'DataStructures' / 'COG' / 'cog_categories.json'
cog_categories_path.parent.mkdir(parents=True, exist_ok=True)
cog_category_rows = [
    {"a_id": cat, "a_name": cat_names.get(cat, cat), "b_id": cog_id, "b_name": entry["name"]}
    for cog_id, entry in cog_map.items()
    for cat in entry["categories"]
]
with open(cog_categories_path, "w", encoding="utf-8") as f:
    json.dump(cog_category_rows, f, indent=2)
print(f"Wrote {len(cog_category_rows)} (category, COG) membership rows to {cog_categories_path}")

cog_map: 5061 entries (400 in more than one category)
cat_names: 26 categories
  A: RNA processing and modification
  B: Chromatin structure and dynamics
  C: Energy production and conversion
  D: Cell cycle control, cell division, chromosome partitioning
  E: Amino acid transport and metabolism
  F: Nucleotide transport and metabolism
  G: Carbohydrate transport and metabolism
  H: Coenzyme transport and metabolism
  I: Lipid transport and metabolism
  J: Translation, ribosomal structure and biogenesis
  K: Transcription
  L: Replication, recombination and repair
  M: Cell wall/membrane/envelope biogenesis
  N: Cell motility
  O: Posttranslational modification, protein turnover, chaperones
  P: Inorganic ion transport and metabolism
  Q: Secondary metabolites biosynthesis, transport and catabolism
  R: General function prediction only
  S: Function unknown
  T: Signal transduction mechanisms
  U: Intracellular trafficking, secretion, and vesicular transport
  V: Defense mechanisms
  W

## 3. Read COG annotation files → cog_db

`cog_db[gene_id]` holds parallel lists: one entry per observation of that gene across all samples.

**Fix:** both branches of the if/else write to the same `cog_db` parameter (previously the else branch accidentally wrote to the outer `cog_db` variable, dropping multi-sample entries).

In [4]:
def read_cog(directpath, filename, s, cog_db):
    """
    Parse one sample's .a.cog.txt and append each gene observation
    to cog_db.  The three lists (lookupId, timepoint, scaffolds) are
    parallel: index i of each belongs to the same observation.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            cog_id  = parts[1]
            # Same idea as the one-id-per-entry assert in
            # EGC_KEGG_TREE/EGC_ENZYME_TREE: .a.cog.txt is always 10
            # columns with a single COG + 4-digit id (scanned 2026-10-14: all
            # 53.7M real rows across 128 samples). A ';'-joined id list or a
            # shifted column fails here loudly instead of building a wrong tree.
            assert len(parts) == 10, (
                f"Expected 10 columns in {path}, got {len(parts)} "
                f"(gene {gene_id}) -- the raw .a.cog.txt format may have changed."
            )
            assert cog_id.startswith("COG") and len(cog_id) == 7 and cog_id[3:].isdigit(), (
                f"Expected exactly one COG id (COG + 4 digits), got {cog_id!r} "
                f"(gene {gene_id}) in {path} -- the raw .a.cog.txt format may have changed."
            )
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in cog_db:
                cog_db[gene_id] = {
                    'lookupId':  [cog_id],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                cog_db[gene_id]['lookupId'].append(cog_id)  
                cog_db[gene_id]['timepoint'].append(s)
                cog_db[gene_id]['scaffolds'].append(scaff)


base   = REPO / 'Metagenome_Samples'
cog_db = {}

for s in samples:
    read_cog(base / s, f"{s}.a.cog.txt", s, cog_db)

print(f"cog_db: {len(cog_db):,} genes")

cog_db: 50,070,928 genes


## 4. Build COG tree

COG has no deep biological hierarchy the way KEGG/Enzyme do -- just two
levels: the 26 single-letter functional categories (C, J, K, …), and the
specific COG ids under them. No gene is ever directly annotated with a bare
category letter (confirmed against real `.a.cog.txt` files -- every
observation is a specific COG####), so category-level EGC is entirely
derived by rollup, the same principle as every other intermediate node in
this whole pipeline.

A COG id can belong to more than one of NCBI's 26 categories (confirmed: 400 of ~5,000
COGs, e.g. COG0028 is listed on both category E's and H's pages). Each gene
is registered at its specific COG id (depth=2) **and every one of that
COG's real categories** (depth=1) -- not just one. The depth=2 leaf still
displays a single primary category for now (alphabetically first, for a
reproducible choice) since its own EGC and JGI-comparison value don't
depend on category count at all; only depth=1 category totals change.

**Deduplication:** hmmsearch reports one row per domain alignment hit, so a
multi-domain protein can appear multiple times for the same COG in the same
sample. Each gene is counted **once per unique gene_id per (node, sample)**
using the `genes` set as a gate, at every node it's registered to. This
matches JGI's unique-gene counting and also means a gene isn't double
counted within a single category even though it may now reach that
category via the multi-category walk.

In [5]:
cog_tree = {}  # {node_id: {name, category, depth, parent_id, parent_name, timepoints}}
dropped_cog_ids = set()  # real COG ids absent from NCBI's scraped cog_map

def _register(node_id, name, category, depth, parent_id, parent_name, sample, gene_id, scaff):
    if node_id not in cog_tree:
        cog_tree[node_id] = {
            'name':        name,
            'category':    category,
            'depth':       depth,
            'parent_id':   parent_id,
            'parent_name': parent_name,
            'timepoints':  {},
        }
    node = cog_tree[node_id]
    if sample not in node['timepoints']:
        node['timepoints'][sample] = {'egc': 0, 'norm_egc': 0.0, 'scaffolds': {}, 'genes': set()}
    tp = node['timepoints'][sample]
    if gene_id not in tp['genes']:
        tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
    tp['genes'].add(gene_id)


for gene_id, ginfo in cog_db.items():
    cog_ids    = ginfo['lookupId']
    timepoints = ginfo['timepoint']
    scaffolds  = ginfo['scaffolds']

    for cog_id, sample, scaff in zip(cog_ids, timepoints, scaffolds):
        if cog_id not in cog_map:
            # A truncated NCBI scrape is indistinguishable
            # from a genuinely obsolete id -- can't tell which without a real
            # count. Skip is still the right behavior (can't classify what
            # isn't in the reference), just make it visible instead of silent.
            dropped_cog_ids.add(cog_id)
            continue

        cog_name   = cog_map[cog_id]['name']
        # A COG can be listed on more than
        # one of NCBI's category pages -- sorted for a reproducible choice
        # of which one the leaf node displays as "the" category below.
        categories = sorted(cog_map[cog_id]['categories'])
        primary_cat_letter = categories[0]
        primary_cat_name   = cat_names.get(primary_cat_letter, primary_cat_letter)

        # Leaf: the specific COG id, registered once regardless of how many
        # categories it belongs to (the 'genes' dedup guard means calling
        # this again for the same gene/sample is already a no-op, but only
        # one call is made here to keep it explicit). category/parent_id/
        # parent_name display the primary category only -- see cell 4's
        # note above; this does not affect the leaf's own EGC.
        _register(cog_id, cog_name, primary_cat_name, 2, primary_cat_letter, primary_cat_name,
                   sample, gene_id, scaff)

        # Category: register this gene at EVERY real category the COG
        # belongs to, not just the primary one. The per-node 'genes' set still guards each category
        # individually, so a gene reaching the same category twice (it
        # can't, here) or being processed once per COG is still counted
        # once per category, never inflated.
        for cat_letter in categories:
            cat_name = cat_names.get(cat_letter, cat_letter)
            _register(cat_letter, cat_name, cat_name, 1, None, None, sample, gene_id, scaff)

print(
    f"cog_tree: {len(cog_tree)} nodes "
    f"({sum(1 for n in cog_tree.values() if n['depth'] == 1)} categories, "
    f"{sum(1 for n in cog_tree.values() if n['depth'] == 2)} COG ids)"
)
print(f"Dropped {len(dropped_cog_ids)} distinct COG ids not found in NCBI's cog_map (see docs/MISSING_ANNOTATIONS.md)")
if dropped_cog_ids:
    print(f"  e.g. {sorted(dropped_cog_ids)[:10]}")

cog_tree: 4637 nodes (25 categories, 4612 COG ids)
Dropped 19 distinct COG ids not found in NCBI's cog_map (see docs/MISSING_ANNOTATIONS.md)
  e.g. ['COG5022', 'COG5032', 'COG5038', 'COG5044', 'COG5049', 'COG5074', 'COG5096', 'COG5183', 'COG5239', 'COG5244']


## 5. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a COG (from this notebook's own `cog_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_KEGG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a COG hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a COG-only subset of the shared total) can never exceed it.

In [6]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_KEGG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = cog_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in cog_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 6. Write .cog.json files

In [8]:
base = REPO / 'DataStructures' / 'COG'
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for node_id, node in cog_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_id] = {
            'name':        node['name'],
            'category':    node['category'],
            'depth':       node['depth'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp.get('egc',       0),
            'norm_egc':    tp.get('norm_egc',  0.0),
            'scaffolds':   tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.cog.json", 'w', encoding='utf-8') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .cog.json files to {base}")

Wrote 128 .cog.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\COG
